In [15]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free

CASH = 50_000
COMMISSION = 0.00015
TABLE_NAME = "f0_smoothing_chandelier_exit"
SMOOTH_PERIOD = 12

def load_data(file_path, *, verbose=False, use_vwap=False):
    # Compact float32 + Parquet-cached loader.
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def make_signals(data, n1=9, atr_period=14, sl_mult=1.0, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    # keep float32 through indicator math; cast up only at the vectorbt boundary
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    smoothed = vbt.indicators.MA.run(ema1.ma, window=smooth_period, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    sl_pct = sl_dist / close

    long_mask = (smoothed.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema1.ma > smoothed.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])
    long_signal = long_mask
    short_signal = short_mask

    return pd.DataFrame({
        "ema1": ema1.ma,
        "smoothed": smoothed.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "long": long_signal,
        "short": short_signal,
    })

def build_chandelier_exits(data, signals, trail_mult):
    atr_np = signals["atr"].to_numpy(dtype=np.float64)
    high_np = data["High"].to_numpy(dtype=np.float64)
    low_np  = data["Low"].to_numpy(dtype=np.float64)
    long_in  = signals["long"].to_numpy(dtype=bool)
    short_in = signals["short"].to_numpy(dtype=bool)
    n = len(atr_np)

    long_exits  = np.zeros(n, dtype=bool)
    short_exits = np.zeros(n, dtype=bool)

    peak = np.nan
    ref  = np.nan
    pos  = 0
    for i in range(n):
        if pos == 0:
            if long_in[i] and np.isfinite(atr_np[i]) and atr_np[i] > 0:
                pos  = 1
                peak = high_np[i]
                ref  = atr_np[i]
            elif short_in[i] and np.isfinite(atr_np[i]) and atr_np[i] > 0:
                pos  = -1
                peak = low_np[i]
                ref  = atr_np[i]
            continue
        if pos == 1:
            peak = max(peak, high_np[i])
            trail = peak - trail_mult * ref
            if low_np[i] <= trail:
                long_exits[i] = True
                pos = 0
                peak = np.nan
                ref = np.nan
        else:
            peak = min(peak, low_np[i])
            trail = peak + trail_mult * ref
            if high_np[i] >= trail:
                short_exits[i] = True
                pos = 0
                peak = np.nan
                ref = np.nan

    return long_exits, short_exits

def run_backtest(data, n1=9, atr_period=14, sl_mult=1.0, trail_mult=2.0, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    signals = make_signals(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, use_vwap=use_vwap, smooth_period=smooth_period)
    freq = data.index.to_series().diff().median()

    long_signal  = signals["long"].to_numpy()
    short_signal = signals["short"].to_numpy()
    # vbt wants float64 for size/close; cast just the columns it needs.
    close_np = data["Close"].to_numpy(dtype=np.float64)

    long_exits, short_exits = build_chandelier_exits(data, signals, trail_mult)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=long_exits,
        short_entries=short_signal,
        short_exits=short_exits,
        init_cash=CASH,
        fees=COMMISSION,
        size=100.0,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=None,
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, atr_period, sl_mult, trail_mult, use_vwap=False, asset=None, timeframe=None, smooth_period=SMOOTH_PERIOD):
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, trail_mult=trail_mult, use_vwap=use_vwap, smooth_period=smooth_period)
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "trail_mult": trail_mult,
            "use_vwap": use_vwap,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "trail_mult": trail_mult,
            "use_vwap": use_vwap,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [5]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M1.csv").expanduser()
data = load_data(file_path, verbose=True)
data = data[data.index >= data.index[-1] - pd.Timedelta(days=7)]

portfolio, stats, signals = run_backtest(data, trail_mult=2.0)
print(stats)

trades = portfolio.trades.records_readable
print(trades)

free(data, signals)

[loader] cache hit: XAUUSD_M1__cache_v1.parquet
Start                                                 0
End                                                5495
Period                                  3 days 19:36:00
Start Value                                     50000.0
End Value                                  49977.680163
Total Return [%]                               -0.04464
Benchmark Return [%]                          -3.831117
Max Gross Exposure [%]                         0.200813
Total Fees Paid                               23.084517
Max Drawdown [%]                               0.045206
Max Drawdown Duration                   3 days 18:35:00
Total Trades                                        770
Total Closed Trades                                 769
Total Open Trades                                     1
Open Trade PnL                                   -0.015
Win Rate [%]                                  42.262679
Best Trade [%]                                 0.414758


In [ ]:
portfolio.plot()

## Optimization

In [13]:

from itertools import product
import time


db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    trail_mult REAL NOT NULL,
    use_vwap INTEGER NOT NULL,
    smooth_period INTEGER NOT NULL,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15, 2), # fast ema,
    range(6, 16, 2), # smoothing
    [14, 18, 21], # atr period
    [1.0, 2.0, 3.0], # sl mult
    [1.0, 2.0, 3.0], # trail mult
    [False, True], # use vwap
))

#* Only for test
# grid = list(product(
#     ["M1"], # timeframes
#     [9], # fast ema
#     [12], # smoothing
#     [14], # atr period
#     [1], # sl mult
#     [2.0], # trail mult
#     [True, False], # use vwap
# ))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
# Load each timeframe once and reuse across all param combos (big memory win on M1).
data_by_tf: dict[tuple[str, bool], pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    for uv in sorted({g[6] for g in grid if g[0] == tf}):
        fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
        data_by_tf[(tf, uv)] = load_data(fp, verbose=True, use_vwap=uv)

ONE_LOOP_MEAN_TIME = 0.0
for i, (timeframe, n1, smoothing, ap, sm, tm, uv) in enumerate(grid):

    data = data_by_tf[(timeframe, uv)]

    t1 = time.time()
    result = run_params(data, n1, ap, sm, tm, use_vwap=uv, asset=asset, timeframe=timeframe, smooth_period=smoothing)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, atr_period, sl_mult, trail_mult, use_vwap, smooth_period, asset, timeframe, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["atr_period"], result["sl_mult"], result["trail_mult"], uv, result["smooth_period"], result["asset"], result["timeframe"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    ONE_LOOP_MEAN_TIME = (elapsed + ONE_LOOP_MEAN_TIME * i) / (i + 1) if i else elapsed
    combinations_left = len(grid) - (i + 1)
    est_left_sec = ONE_LOOP_MEAN_TIME * combinations_left
    if (i + 1) % max(1, len(grid) // 100) == 0 or (i + 1) == len(grid):
        print(
            f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} ap={ap} sm={sm} tm={tm} vwap={uv} "
            f"| sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s) "
            f"| mean={ONE_LOOP_MEAN_TIME:.1f}s est_left={est_left_sec/60:.1f}min"
        )

    free()

free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)



[11:21:07] Testing 4050 combinations -> f0_smoothing_chandelier_exit
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[11:22:19]    40/4050 tf=M1 n1=5 ap=21 sm=1.0 tm=2.0 vwap=True | sharpe=-43.910178808907816 ret=-6.094154355076533 trades=103421 (1.3s) | mean=1.6s est_left=106.8min
[11:23:30]    80/4050 tf=M1 n1=5 ap=18 sm=2.0 tm=1.0 vwap=True | sharpe=-52.15475961340229 ret=-6.384967885144811 trades=110274 (1.4s) | mean=1.6s est_left=105.2min
[11:24:40]   120/4050 tf=M1 n1=5 ap=14 sm=2.0 tm=3.0 vwap=True | sharpe=-16.201320330429763 ret=-2.3927150986131656 trades=41040 (1.3s) | mean=1.6s est_left=103.4min
[11:25:51]   160/4050 tf=M1 n1=5 ap=21 sm=3.0 tm=2.0 vwap=True | sharpe=-28.984058479080645 ret=-3.9194275933474723 trades=68120 (1.4s) | mean=1

,id,n1,atr_period,sl_mult,trail_mult,use_vwap,smooth_period,asset,timeframe,expectancy,win_rate,max_dd,sharpe_ratio,profit_factor,num_trades,return_pct
0,2898,5,18,3.0,3.0,1,12,XAUUSD,M15,-0.012301,34.095046,0.163812,-0.943449,0.916840,4946,-0.121106
1,2880,5,14,3.0,3.0,1,12,XAUUSD,M15,-0.012888,33.760684,0.171959,-0.975588,0.913064,4915,-0.126110
2,2874,5,14,2.0,3.0,1,12,XAUUSD,M15,-0.012254,32.544494,0.176016,-0.980800,0.913758,5114,-0.124755
3,2916,5,21,3.0,3.0,1,12,XAUUSD,M15,-0.012779,34.164990,0.163911,-0.992822,0.913531,4971,-0.126470
4,2892,5,18,2.0,3.0,1,12,XAUUSD,M15,-0.012538,32.918530,0.174320,-1.018659,0.911824,5144,-0.128407
5,3168,7,18,3.0,3.0,1,12,XAUUSD,M15,-0.012828,32.578397,0.169723,-1.034240,0.910733,5167,-0.131989
6,3186,7,21,3.0,3.0,1,12,XAUUSD,M15,-0.012780,32.786885,0.168343,-1.039919,0.911088,5186,-0.131976
7,2970,5,21,3.0,3.0,1,14,XAUUSD,M15,-0.013249,33.126128,0.173352,-1.044895,0.909128,4988,-0.131593
8,3816,13,18,3.0,3.0,1,6,XAUUSD,M15,-0.013234,32.690795,0.172744,-1.054375,0.908585,5085,-0.134014
9,3834,13,21,3.0,3.0,1,6,XAUUSD,M15,-0.013233,32.960565,0.173380,-1.063907,0.908607,5098,-0.134343


In [14]:
vwap_comparison = pd.read_sql_query(
    f"""SELECT use_vwap,
              COUNT(*) AS n,
              AVG(expectancy) AS mean_expectancy,
              AVG(win_rate) AS mean_win_rate,
              AVG(max_dd) AS mean_max_dd,
              AVG(sharpe_ratio) AS mean_sharpe,
              AVG(profit_factor) AS mean_profit_factor,
              AVG(num_trades) AS mean_num_trades,
              AVG(return_pct) AS mean_return_pct
       FROM {TABLE_NAME}
       WHERE sharpe_ratio IS NOT NULL
       GROUP BY use_vwap""",
    conn,
)
print(vwap_comparison)


   use_vwap     n  mean_expectancy  mean_win_rate  mean_max_dd  mean_sharpe  \
0         0  2025        -0.029157      47.971807     6.034179   -37.868356   
1         1  2025        -0.026545      30.427113     1.898985   -15.206586   

   mean_profit_factor  mean_num_trades  mean_return_pct  
0            0.569467    104702.850864        -6.032046  
1            0.638994     32874.030123        -1.889997  


In [ ]:
n1_comparison = pd.read_sql_query(
    """
    SELECT n1,
           COUNT(*) AS n,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL
    GROUP BY n1
    ORDER BY n1
    """,
    conn,
)
print(n1_comparison)

In [ ]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)


n1_comparison_m5 = pd.read_sql_query(
    """
    SELECT n1,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL AND timeframe like 'M5'
    GROUP BY n1
    ORDER BY mean_win_rate, mean_expectancy
    """,
    conn,
)
print(n1_comparison_m5)